<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Context

Planar camera calibration is required to recover the geometric mapping between 3-D scene coordinates and 2-D image measurements. In this laboratory, a calibrated pinhole model is estimated from multiple views of a planar chessboard with known metric geometry.

The experimental challenge is not limited to estimating a matrix. Reliable calibration depends on accurate point correspondences, numerically conditioned homography estimation, consistent multi-view constraints, physically valid camera poses, and residual analysis capable of exposing model inadequacy or poorly conditioned observations.

## Problem Statement

Estimate the intrinsic matrix $K$ and one camera pose $(R,t)$ for each valid chessboard view using normalized DLT and Zhang's closed-form planar calibration method.

The calibration must:

- detect and refine image-space corner observations;
- construct metrically consistent planar coordinates;
- estimate one homography per retained view under numerical normalization;
- recover camera intrinsics from the multi-view Zhang constraints;
- recover geometrically valid rotations and translations;
- reproject all calibration points into the image plane;
- quantify per-view and global reprojection error;
- produce diagnostic figures sufficient to assess geometric and numerical consistency.

Lens-distortion estimation, nonlinear bundle adjustment, and robust outlier rejection are outside the defined scope.

## Inputs and Fixed Parameters

| Item | Fixed value / convention |
| --- | --- |
| Input directory | `../data/calibration_images/` |
| Input format | JPEG (`*.jpg`) |
| Calibration target | planar chessboard |
| Internal-corner pattern | $8\times6$ |
| Points per valid view | $48$ |
| Physical square size | $0.03\,\mathrm{m}$ |
| Calibration plane | $Z=0$ |
| Minimum retained views | $3$ |
| Homography estimation | normalized DLT + SVD |
| Intrinsic estimation | Zhang closed-form constraints + SVD |
| Pose model | pinhole camera, no lens distortion |
| Rotation correction | projection to the nearest proper rotation matrix |
| Primary validation quantity | reprojection error in pixels |
| Required outputs | six diagnostic PNG figures under `outputs/figures/` |

All paths must be repository-relative. Views with incomplete chessboard detection are excluded explicitly rather than silently padded or approximated.

## 1. Load the Sorted JPEG Calibration Images

1. Enumerate every `*.jpg` file in the calibration directory and sort the filenames deterministically.
2. Verify that the directory exists and that at least one calibration image is available.
3. Read each image and record:
   - filename;
   - width and height;
   - channel count;
   - load status.
4. Reject unreadable files with an explicit diagnostic.
5. Report the number of candidate views before corner detection.

**Required evidence**

- deterministic ordered image list;
- image-count summary;
- image-size consistency check.

No calibration parameter is estimated at this stage; the purpose is to establish a reproducible experimental dataset.

## 2. Detect and Refine Chessboard Corners

For each candidate image:

1. convert the image to grayscale;
2. detect the complete $8\times6$ internal-corner pattern;
3. reject the view if the complete pattern cannot be recovered;
4. refine the detected corner coordinates to sub-pixel precision;
5. retain the refined image coordinates in the same indexing order for every view.

For one valid view, let the measured image points be

$$
\mathbf{x}_i=
\begin{bmatrix}
u_i\\
v_i
\end{bmatrix},
\qquad i=1,\ldots,48.
$$

The detector/refinement configuration used in the implementation must be documented.

**Required evidence**

- number of valid and rejected views;
- exactly 48 refined corners for every retained view;
- one diagnostic figure showing the detected/refined chessboard corners;
- explicit confirmation that corner ordering is consistent across views.

## 3. Build the Planar World Coordinates

Construct the known calibration geometry in metric units.

For square size

$$
s_q=0.03\,\mathrm{m},
$$

generate the $48$ planar points

$$
\mathbf{X}_{p,i}=
\begin{bmatrix}
X_i\\
Y_i\\
1
\end{bmatrix},
$$

with:

- the first internal corner at $(0,0)$;
- the $X$ axis along the 8-corner direction;
- the $Y$ axis along the 6-corner direction;
- $Z=0$ for every physical calibration point.

The maximum coordinates must therefore be consistent with the $8\times6$ grid and the $0.03\,\mathrm{m}$ square size.

**Required evidence**

- array shape $48\times2$ or equivalent homogeneous representation;
- first and last few coordinates printed or tabulated;
- consistent point ordering with the detected image corners.

## 4. Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$

Normalize image points and planar points independently using a similarity transform.

For a generic 2-D point set $\{(x_i,y_i)\}_{i=1}^{N}$, compute the centroid

$$
\bar{x}=\frac{1}{N}\sum_i x_i,
\qquad
\bar{y}=\frac{1}{N}\sum_i y_i,
$$

and the mean Euclidean distance to the centroid

$$
\bar d=
\frac{1}{N}
\sum_i
\sqrt{(x_i-\bar x)^2+(y_i-\bar y)^2}.
$$

Use the scale

$$
s=\frac{\sqrt{2}}{\bar d}
$$

and construct

$$
T=
\begin{bmatrix}
s&0&-s\bar x\\
0&s&-s\bar y\\
0&0&1
\end{bmatrix}.
$$

Apply this procedure separately to the image coordinates and the planar coordinates.

**Validation**

After transformation, verify numerically that each normalized point set has:

- centroid approximately $(0,0)$;
- mean distance approximately $\sqrt{2}$.

Report the normalization scales and confirm that the normalized DLT system is better conditioned than the unnormalized formulation.

## 5. Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD

For every retained view, estimate the plane-to-image homography from all $48$ normalized correspondences.

For normalized planar point $(X_i,Y_i)$ and normalized image point $(u_i,v_i)$, add the two DLT rows

$$
\begin{bmatrix}
X_i&Y_i&1&0&0&0&-u_iX_i&-u_iY_i&-u_i
\end{bmatrix},
$$

$$
\begin{bmatrix}
0&0&0&X_i&Y_i&1&-v_iX_i&-v_iY_i&-v_i
\end{bmatrix}.
$$

Thus, with 48 correspondences,

$$
Q\in\mathbb{R}^{96\times9},
\qquad
Q\mathbf h=0.
$$

Solve the homogeneous system using SVD,

$$
Q=U\Sigma V^T,
$$

and retain the right-singular vector associated with the smallest singular value as $\mathbf h$. Reshape it into the normalized homography $H_n$.

**Required evidence**

- correct DLT matrix dimensions for every retained view;
- finite singular values;
- rank sufficient for a unique homography up to scale;
- one representative diagnostic showing the normalized correspondence/homography estimation stage.

## 6. Denormalize Each Homography

Recover the homography in the original coordinate systems using

$$
H=
T_{\mathrm{image}}^{-1}
H_n
T_{\mathrm{plane}}.
$$

Because a homography is defined only up to a non-zero scale, normalize it such that

$$
H_{33}=1
$$

whenever this value is numerically safe to use.

For each retained view:

1. verify that all entries of $H$ are finite;
2. verify that $H$ maps the planar calibration points to plausible image coordinates;
3. report or store the resulting $3\times3$ matrix.

Degenerate homographies must be rejected explicitly rather than propagated into Zhang calibration.

## 7. Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD

For each homography

$$
H=
\begin{bmatrix}
\mathbf h_1&\mathbf h_2&\mathbf h_3
\end{bmatrix},
$$

use the orthonormality of the first two rotation columns to form Zhang's constraints.

Define

$$
B=K^{-T}K^{-1}
$$

and, for homography columns $\mathbf h_i$ and $\mathbf h_j$, construct the standard vector $v_{ij}$ such that

$$
\mathbf h_i^TB\mathbf h_j
=
v_{ij}^Tb.
$$

Each valid view contributes

$$
v_{12}^Tb=0,
$$

and

$$
(v_{11}-v_{22})^Tb=0.
$$

Stack all constraints into

$$
Vb=0
$$

and solve by SVD.

**Required evidence**

- $V$ contains two rows per retained view;
- at least three geometrically distinct retained views are used;
- the smallest-singular-vector solution is finite;
- the sign ambiguity of the homogeneous vector $b$ is handled consistently during intrinsic recovery.

## 8. Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$

Recover the camera intrinsic parameters from

$$
b=
[B_{11},B_{12},B_{22},B_{13},B_{23},B_{33}]^T.
$$

Compute the Zhang closed-form quantities required to obtain:

- horizontal focal scale $\alpha$;
- vertical focal scale $\beta$;
- skew $\gamma$;
- principal point $(u_0,v_0)$.

Assemble

$$
K=
\begin{bmatrix}
\alpha&\gamma&u_0\\
0&\beta&v_0\\
0&0&1
\end{bmatrix}.
$$

**Validation**

The retained intrinsic matrix must satisfy:

- $\alpha>0$ and $\beta>0$;
- finite entries;
- bottom row equal to $[0,0,1]$ within numerical precision;
- real-valued square-root terms;
- a principal point and skew that are physically plausible for the image dimensions.

Report the final intrinsic matrix numerically.

## 9. Recover $R$ and $t$ for Every Retained View

For each retained homography $H=[\mathbf h_1\ \mathbf h_2\ \mathbf h_3]$, compute

$$
\lambda_p=
\frac{1}{\|K^{-1}\mathbf h_1\|_2}.
$$

Recover preliminary pose components

$$
\mathbf r_1=
\lambda_pK^{-1}\mathbf h_1,
\qquad
\mathbf r_2=
\lambda_pK^{-1}\mathbf h_2,
$$

$$
\mathbf r_3=
\mathbf r_1\times\mathbf r_2,
\qquad
t=
\lambda_pK^{-1}\mathbf h_3.
$$

Form the approximate rotation matrix and project it onto the nearest proper rotation using SVD.

The final pose must satisfy

$$
R^TR\approx I,
\qquad
\det(R)\approx+1.
$$

**Required evidence**

- one $(R,t)$ pair per retained view;
- numerical orthonormality and determinant checks;
- camera-pose visualization derived from the recovered extrinsics.

## 10. Reproject the $Z=0$ Calibration Points

For every planar calibration point,

$$
\mathbf X_w=
[X,Y,0]^T,
$$

compute its camera-frame coordinate

$$
\mathbf X_c=
R\mathbf X_w+t,
$$

then project through the intrinsic matrix

$$
\tilde{\mathbf x}
=
K\mathbf X_c.
$$

Convert homogeneous coordinates to pixels:

$$
\hat u=
\frac{\tilde u}{\tilde w},
\qquad
\hat v=
\frac{\tilde v}{\tilde w}.
$$

For every retained view, superimpose measured and reprojected corners and verify that all projected coordinates are finite and geometrically plausible.

## 11. Compute Point-wise Errors, Mean Error and RMSE

For measured point $\mathbf x_i=(u_i,v_i)$ and reprojected point $\hat{\mathbf x}_i=(\hat u_i,\hat v_i)$, compute the Euclidean reprojection error

$$
e_i=
\sqrt{
(u_i-\hat u_i)^2+
(v_i-\hat v_i)^2
}.
$$

For each retained view, report:

$$
\bar e=
\frac{1}{N}
\sum_{i=1}^{N}e_i,
$$

and

$$
\mathrm{RMSE}
=
\sqrt{
\frac{1}{N}
\sum_{i=1}^{N}e_i^2
}.
$$

Also report overall mean error and overall RMSE across all retained views.

**Required analysis**

- identify the best and worst calibrated views by reprojection error;
- inspect whether large residuals are isolated or systematic;
- discuss whether border residuals could indicate the effect of omitted lens distortion.

## 12. Produce and Save the Six Required Diagnostic Figures

Generate and save exactly the following figures under `outputs/figures/`:

1. `detected_chessboard_corners.png`  
   Show at least one retained calibration view with detected/refined corner locations.

2. `homography_estimation_pipeline.png`  
   Visualize representative planar/image correspondences and the homography-estimation stage.

3. `estimated_camera_poses.png`  
   Display the recovered camera/target geometry for the retained views.

4. `reprojection_results.png`  
   Overlay measured and reprojected calibration points.

5. `mean_reprojection_error_by_view.png`  
   Plot the mean reprojection error for every retained view.

6. `reprojection_error_distribution.png`  
   Show the distribution of point-wise reprojection errors.

Every figure must include readable labels/titles and be reproducible from the notebook execution.

## 13. Run the Numerical and Output-file Validation Checks

Before accepting the calibration, verify all of the following:

1. at least three valid chessboard views are retained;
2. every retained view contains exactly 48 image/planar correspondences;
3. every homography is finite and $3\times3$;
4. $K$ is finite, $3\times3$, and physically admissible;
5. every recovered $R$ satisfies $R^TR\approx I$;
6. every recovered $R$ satisfies $\det(R)\approx1$;
7. all translation vectors are finite;
8. all reprojected points and error values are finite;
9. all six required diagnostic figures exist;
10. no rejected view is used in the intrinsic or pose solution.

Any failed condition must stop the final validation and identify the offending quantity.

## Completion Criterion

The laboratory is complete when the full calibration can be reproduced from the supplied images without manual intervention after execution begins, all retained views satisfy the geometric and numerical checks above, the intrinsic/extrinsic parameters are reported, the reprojection analysis is complete, and the six required diagnostic figures have been generated successfully.